In [1]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "sivakumar",
    database = "cart2insights"
)
cursor = conn.cursor()
print("connection successful")

connection successful


In [12]:
query_orders = """
SELECT
    o.order_id,

    SUM(oi.price + oi.freight_value) AS total_order_value,

    DATEDIFF(
        o.order_delivered_customer_date,
        o.order_purchase_timestamp
    ) AS delivery_days,

    DATEDIFF(
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date
    ) AS delivery_delay

FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id

GROUP BY
    o.order_id,
    o.order_delivered_customer_date,
    o.order_purchase_timestamp,
    o.order_estimated_delivery_date;
"""

In [ ]:
orders_features = pd.read_sql(query_orders, conn)

orders_features.head()

C:\Users\yendl\AppData\Local\Temp\ipykernel_23848\571147859.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql(query_orders, conn)


,order_id,total_order_value,delivery_days,delivery_delay
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,7.0,-9.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,16.0,-3.0
2,000229ec398224ef6ca0657da4fc703e,216.87,8.0,-14.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,6.0,-6.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,25.0,-16.0


In [ ]:
orders_features.to_csv(
    "../data/features/orders.csv",
    index=False
)

In [37]:
query_customers = """
SELECT
    o.customer_id,

    COUNT(o.customer_id) AS customer_order_count,

    SUM(oi.price + oi.freight_value) AS customer_total_spending,

    AVG(order_total.total_value) AS average_order_value,

    CASE
        WHEN COUNT(o.customer_id) > 1 THEN 'Yes'
        ELSE 'No'
    END AS repeat_customer

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN (
    SELECT
        order_id,
        SUM(price + freight_value) AS total_value
    FROM order_items
    GROUP BY order_id
) order_total
    ON o.order_id = order_total.order_id

GROUP BY o.customer_id;
"""

In [ ]:
customers_features = pd.read_sql(query_customers,conn)

customers_features.head()

C:\Users\yendl\AppData\Local\Temp\ipykernel_23848\587042992.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql(query_customers,conn)


,customer_id,customer_order_count,customer_total_spending,average_order_value,repeat_customer
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74,No
1,000161a058600d5901f007fab4c27140,1,67.41,67.41,No
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42,No
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35,No
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01,No


In [ ]:
customers_features.to_csv(
    "../data/features/customers.csv",
    index=False
)

In [21]:
query_sellers = """
SELECT
    seller_id,

    SUM(price) AS seller_revenue,

    COUNT(DISTINCT order_id) AS seller_order_count

FROM order_items

GROUP BY seller_id;
"""

In [ ]:
sellers_features = pd.read_sql(query_sellers,conn)

sellers_features.head()

C:\Users\yendl\AppData\Local\Temp\ipykernel_23848\399795121.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sellers = pd.read_sql(query_sellers,conn)


,seller_id,seller_revenue,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03,200
2,001e6ad469a905060d959994f1b41e4f,250.00,1
3,002100f778ceb8431b7a1020ff7ab48f,1234.50,51
4,003554e2dce176b5555353e4f3555ac8,120.00,1


In [ ]:
sellers_features.to_csv(
    "../data/features/sellers.csv",
    index=False
)